# Construirea aplicațiilor de generare a imaginilor (OpenAI)

LLM-urile fac mai mult decât generarea de text. Poți genera și imagini din descrieri textuale. Imaginile ca modalitate sunt utile în domenii precum MedTech, arhitectură, turism, dezvoltarea jocurilor, marketing și altele. În această lecție lucrăm cu modelele **GPT Image** de astăzi de pe platforma OpenAI.

## Obiectivele învățării

Până la finalul acestei lecții vei putea:

- Explica ce este generarea de imagini și unde este utilă.
- Înțelege familia de modele `gpt-image` și cum diferă de modelele DALL·E clasice.
- Construiește o aplicație de generare a imaginilor și editează imagini.

## Ce este generarea de imagini?

Modelele de generare a imaginilor creează imagini pornind de la un prompt text. Modelele moderne precum `gpt-image` învață relația dintre text și imagini în timpul antrenamentului, apoi transformă iterativ zgomotul aleatoriu într-o imagine care se potrivește cu promptul tău.

Două familii cunoscute de modele de imagini sunt:

- **`gpt-image` (OpenAI)** — generația curentă folosită în această lecție. Suportă generarea de imagini din text și editarea imaginilor (inpainting cu mască).
- **Midjourney** — un model popular terț cu propriul serviciu și flux de lucru bazat pe Discord.

> Modelele mai vechi OpenAI pentru imagini — **DALL·E 2** și **DALL·E 3** — sunt clasice. DALL·E 3 nu mai este disponibil pentru noi implementări, iar funcții precum `create_variation` au existat doar în DALL·E 2. Folosește modelele `gpt-image` pentru aplicații noi.

> **Important:** Modelele `gpt-image` returnează imaginea generată ca **base64** (`b64_json`), nu ca URL. Codul tău decodează șirul base64 în octeți și îl salvează — nu există un URL pentru a descărca imaginea.


## Construirea primei tale aplicații de generare a imaginilor

Deci, ce este necesar pentru a construi o aplicație de generare a imaginilor? Ai nevoie de următoarele biblioteci:

- **python-dotenv**, ți se recomandă insistent să folosești această bibliotecă pentru a păstra secretele tale într-un fișier *.env* separat de cod.
- **openai**, această bibliotecă este ceea ce vei folosi pentru a interacționa cu API-ul OpenAI.
- **pillow**, pentru a lucra cu imagini în Python.


1. Creează un fișier *.env* cu următorul conținut:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. Colectați bibliotecile de mai sus într-un fișier numit *requirements.txt* astfel:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. În continuare, creați un mediu virtual și instalați bibliotecile:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> For Windows, use the following commands to create and activate your virtual environment:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Add the following code in file called *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # import dotenv
    dotenv.load_dotenv()

    # Create OpenAI object (reads OPENAI_API_KEY from your .env)
    client = openai.OpenAI()


    try:
        # Create an image by using the image generation API
        generation_response = client.images.generate(
            model="gpt-image-2",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Enter your prompt text here
            size='1024x1024',
            n=1
        )
        # Set the directory for the stored image
        image_dir = os.path.join(os.curdir, 'images')

        # If the directory doesn't exist, create it
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Initialize the image path (note the filetype should be png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image models return the image as base64 (b64_json), not a URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Display the image in the default image viewer
        image = Image.open(image_path)
        image.show()

    # catch exceptions
    except openai.BadRequestError as err:
        print(err)

    ```

Let's explain this code:

- First, we import the libraries we need, including the OpenAI library, the dotenv library, the base64 module, and the Pillow library.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- After that, we create the client, which reads the API key from your ``.env``.

    ```python
    # Create OpenAI object
    client = openai.OpenAI()
    ```

- Next, we generate the image:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-2",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Enter your prompt text here
        size='1024x1024',
        n=1
    )
    ```

    `gpt-image` models return the image as a **base64** string in `data[0].b64_json`. We decode it to bytes and write it to a file — there's no URL to download.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Lastly, we open the image and use the standard image viewer to display it:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### More details on generating the image

Let's look at the parameters of `images.generate`:

- **model**, is the image model to use (for example `gpt-image-2`).
- **prompt**, is the text prompt used to generate the image. Here it's "Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils".
- **size**, is the size of the generated image (for example `1024x1024`, `1536x1024`, `1024x1536`, or `"auto"`).
- **n**, is the number of images generated. Here we generate one.

> Image models don't take a `temperature` parameter — that's a text-generation control. To get variety, call the API again; to reduce variety, make your prompt more specific.

## Additional capabilities of image generation

You've seen how to generate an image with a few lines of Python. `gpt-image` models can also **edit** an existing image. By providing an image, an optional **mask** (which marks the area to change), and a prompt, you can alter part of an image — for example, add a hat to our bunny.

```python
response = client.images.edit(
  model="gpt-image-2",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# edits are also returned as base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

The base image contains just the rabbit; the final image has the hat on the rabbit.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Declinare a responsabilității**:
Acest document a fost tradus folosind serviciul de traducere AI [Co-op Translator](https://github.com/Azure/co-op-translator). În timp ce ne străduim pentru acuratețe, vă rugăm să rețineți că traducerile automate pot conține erori sau inexactități. Documentul original în limba sa nativă trebuie considerat sursa autorizată. Pentru informații critice, se recomandă traducerea profesională realizată de un om. Nu ne asumăm responsabilitatea pentru eventualele neînțelegeri sau interpretări greșite care decurg din utilizarea acestei traduceri.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
